# GTZAN Music Clustering - Results and Visualizations

This notebook displays the results from clustering experiments.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Add src to path
sys.path.append('../src')

from config import Config

# Configure plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline


## Load Results


In [ ]:
config = Config()

# Load metrics CSV
metrics_path = os.path.join(config.metrics_dir, 'metrics.csv')

if os.path.exists(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    print(f"Loaded {len(metrics_df)} experiment results")
    display(metrics_df)
else:
    print(f"Metrics not found: {metrics_path}")
    print("Run experiments first!")


## Compare Models


In [ ]:
if 'metrics_df' in locals():
    # Plot comparison of key metrics
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    metrics_to_plot = ['silhouette', 'ari', 'nmi', 'purity']
    titles = ['Silhouette Score', 'Adjusted Rand Index', 'Normalized Mutual Information', 'Purity']
    
    for i, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
        ax = axes[i // 2, i % 2]
        
        if metric in metrics_df.columns:
            data = metrics_df.sort_values(metric, ascending=False)
            data.plot(x='experiment_name', y=metric, kind='bar', ax=ax, legend=False)
            ax.set_title(f'{title} (Higher is Better)')
            ax.set_xlabel('Experiment')
            ax.set_ylabel(title)
            ax.tick_params(axis='x', rotation=45)
            ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Show best results
    print("\nBest results by Silhouette Score:")
    best_silhouette = metrics_df.nlargest(5, 'silhouette')[['experiment_name', 'model', 'silhouette', 'ari', 'nmi']]
    display(best_silhouette)
    
    print("\nBest results by ARI:")
    best_ari = metrics_df.nlargest(5, 'ari')[['experiment_name', 'model', 'silhouette', 'ari', 'nmi']]
    display(best_ari)


## Visualizations

Display saved visualization plots from experiments.


In [ ]:
# Display saved plots
figures_dir = config.figures_dir

# List available figures
if os.path.exists(figures_dir):
    figures = [f for f in os.listdir(figures_dir) if f.endswith('.png')]
    print(f"Found {len(figures)} figures\n")
    
    # Display t-SNE plots
    tsne_plots = [f for f in figures if 'tsne' in f]
    for plot in tsne_plots:
        print(f"\n### {plot}")
        display(Image(filename=os.path.join(figures_dir, plot)))
else:
    print(f"Figures directory not found: {figures_dir}")


## Conclusion

Review the metrics and visualizations above to compare different approaches:
- **Baselines**: PCA+KMeans, Direct KMeans, AE+KMeans, Spectral
- **EASY (MLP-VAE)**: VAE on MFCC features
- **MEDIUM (Conv-VAE)**: VAE on log-mel spectrograms + multimodal fusion
- **HARD (CVAE/Beta-VAE)**: Conditional VAE with full multimodal fusion

Key metrics:
- **Silhouette, Calinski-Harabasz, Davies-Bouldin**: Unsupervised quality
- **ARI, NMI, Purity**: Alignment with true genres
